<!-- TARGET: public course-materials repo, sessions/m04/demo.ipynb (full version). Copy to work/m04/ before editing. -->
# Module 4 · Live demo

**Python for Data Science · ESADE MIBA · Wed 4 Nov 2026**

The marketplace operations team asks: *how fast do we deliver, and how often are we late?*
Before we can answer, we need Python's building blocks, then pandas, then a clean look at the
orders table.

## 1. Python through data

Six orders from a tiny shop, the same ones as in the warm-up. A **list of dictionaries**: each
dictionary is one row, each key is a column.

In [ ]:
orders = [
    {"order_id": "o1", "customer_state": "SP", "price": 129.90, "freight": 15.10, "purchased": 2, "delivered": 9, "estimated": 12},
    {"order_id": "o2", "customer_state": "RJ", "price": 49.90, "freight": 12.00, "purchased": 3, "delivered": 20, "estimated": 15},
    {"order_id": "o3", "customer_state": "SP", "price": 89.00, "freight": 8.50, "purchased": 5, "delivered": 10, "estimated": 14},
    {"order_id": "o4", "customer_state": "MG", "price": 199.00, "freight": 22.30, "purchased": 6, "delivered": None, "estimated": 18},
    {"order_id": "o5", "customer_state": "SP", "price": 35.50, "freight": 7.90, "purchased": 8, "delivered": 25, "estimated": 20},
    {"order_id": "o6", "customer_state": "RJ", "price": 74.90, "freight": 11.20, "purchased": 9, "delivered": 16, "estimated": 19},
]
len(orders)

### Lists: order, position, length

In [ ]:
states = ["SP", "RJ", "SP", "MG", "SP", "RJ"]
print(states[0], states[-1], states[1:3], len(states))

### Dictionaries: look up by name, not by position

In [ ]:
first_order = orders[0]
print(first_order["price"])
first_order["review_score"] = 5
first_order.keys()

### Loops: the accumulator pattern

*Your turn (1 min):* before running, say out loud what `total_revenue` will be after the first
pass of the loop.

In [ ]:
total_revenue = 0
for order in orders:
    total_revenue = total_revenue + order["price"]
total_revenue

### `if` inside a loop, and the `None` trap

In [ ]:
late_ids = []
for order in orders:
    if order["delivered"] is not None and order["delivered"] > order["estimated"]:
        late_ids.append(order["order_id"])
late_ids

*Your turn (2 min):* what happens if we drop `order["delivered"] is not None and`? Predict, then I run it.

### Functions: name a rule once, reuse it everywhere

In [ ]:
def is_late(order):
    return order["delivered"] is not None and order["delivered"] > order["estimated"]


def delivery_days(order):
    if order["delivered"] is None:
        return None
    return order["delivered"] - order["purchased"]


[delivery_days(order) for order in orders]

### A group-by, by hand

In [ ]:
revenue_by_state = {}
for order in orders:
    state = order["customer_state"]
    if state not in revenue_by_state:
        revenue_by_state[state] = 0
    revenue_by_state[state] = revenue_by_state[state] + order["price"]
revenue_by_state

### The same table in pandas

A **DataFrame** is a table object. It has **attributes** (data you read: `df.shape`, no
brackets) and **methods** (actions you call: `df.head()`, with brackets). Each loop above becomes
one line.

In [ ]:
import pandas as pd

df = pd.DataFrame(orders)
df.shape

In [ ]:
df.head()

In [ ]:
print(df["price"].sum())
print((df["customer_state"] == "SP").sum())
print(df.loc[df["delivered"] > df["estimated"], "order_id"].tolist())
df.groupby("customer_state")["price"].sum()

Lists and dictionaries keep coming back inside pandas:

- a **list** of column names selects columns, a list of values feeds `isin`;
- a **dictionary** renames columns, old name → new name.

In [ ]:
df[["order_id", "price"]]

In [ ]:
df[df["customer_state"].isin(["RJ", "MG"])]

In [ ]:
df.rename(columns={"freight": "freight_value"}).columns.tolist()

## 2. Meet Olist

A real Brazilian marketplace: small shops sell through Olist on the big online stores. About
99,000 orders, 2016–2018, in nine related tables. We start with one: **orders**.

In [ ]:
from pathlib import Path

RAW = Path("../../data/raw")
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
orders.shape

In [ ]:
orders.head()

In [ ]:
orders.info()

*Observe:* every date column is `object`, i.e. text. pandas does not guess dates for us.
`describe()` summarises the numeric columns only, and there are none.

In [ ]:
orders.describe()

The nine tables, and what one row of each means:

| Table | One row is… | Joins on |
|---|---|---|
| orders | an order | `customer_id` |
| order_items | an item line in an order | `order_id`, `product_id`, `seller_id` |
| order_payments | a payment of an order | `order_id` |
| order_reviews | a review | `order_id` |
| customers | a customer *as recorded on one order* | `customer_id` |
| products | a product | `product_id` |
| sellers | a seller | `seller_id` |
| category translation | a category | `product_category_name` |
| geolocation | a zip prefix location (not used) | `zip_code_prefix` |

## 3. Types and selection

In [ ]:
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=DATE_COLUMNS)
orders.dtypes

Already loaded as text? `pd.to_datetime` converts one column after the fact.

In [ ]:
pd.to_datetime(pd.Series(["2017-11-24 10:15:00", "2018-01-02 08:00:00"]))

### One column, several columns

In [ ]:
orders["order_status"].head(3)

In [ ]:
orders[["order_id", "order_status", "order_purchase_timestamp"]].head(3)

### `loc`: rows and columns by label

In [ ]:
orders.loc[0:2, ["order_id", "order_status"]]

### Boolean masks: filter rows with a condition

In [ ]:
is_delivered = orders["order_status"] == "delivered"
is_delivered.head()

In [ ]:
print(is_delivered.sum(), "delivered of", len(orders))
orders[is_delivered].shape

In [ ]:
orders[orders["order_status"].isin(["canceled", "unavailable"])].shape

*Your turn (2 min):* Black Friday week 2017 is 20–26 November. What goes in the `...`?
`orders["order_purchase_timestamp"].between(...)`

In [ ]:
orders["order_purchase_timestamp"].between("2017-11-20", "2017-11-26").sum()

Is that the whole week? `"2017-11-26"` means midnight, so almost all of Sunday is missing.

In [ ]:
black_friday_week = orders["order_purchase_timestamp"].between("2017-11-20", "2017-11-27")
black_friday_week.sum()

In [ ]:
orders["order_status"].value_counts()

## 4. Missing values are information

In [ ]:
orders.isna().sum()

2,965 orders have no delivery date. Are they broken rows, or is something else going on?

In [ ]:
pd.crosstab(orders["order_status"], orders["order_delivered_customer_date"].isna())

*Observe:* almost every missing delivery date belongs to an order that was never delivered.
The `NaN` is telling us the status. Only 8 "delivered" orders lack a date.

What does a blanket `dropna()` do?

In [ ]:
print(len(orders), "→", len(orders.dropna()))

2,980 rows gone, for reasons we didn't choose: a missing `order_approved_at` is enough. Say
*which* column must be present:

In [ ]:
delivered = orders[orders["order_status"] == "delivered"].dropna(subset=["order_delivered_customer_date"])
print(len(delivered))

### Derived columns

In [ ]:
delivered = delivered.assign(
    delivery_days=(delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]).dt.days
)
delivered["delivery_days"].describe()

**How do you know this number is right?** "Late" means delivered after the estimated date.
But `order_estimated_delivery_date` is a *date* (midnight) and the delivery is a *timestamp*.

In [ ]:
late_by_timestamp = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
late_by_day = delivered["order_delivered_customer_date"].dt.normalize() > delivered["order_estimated_delivery_date"]
print(f"late, comparing timestamps: {late_by_timestamp.mean():.1%}")
print(f"late, comparing days:       {late_by_day.mean():.1%}")

An order delivered at 15:00 on its estimated day: late or on time? The code ran both ways. Only
a business decision tells us which number to report. We use **days**.

In [ ]:
delivered = delivered.assign(is_late=late_by_day)
delivered.sort_values("delivery_days", ascending=False)[
    ["order_id", "order_purchase_timestamp", "order_delivered_customer_date", "delivery_days", "is_late"]
].head()

## Next week

What is *one row* of each table? We save the typed orders to the **clean** layer and meet the
first join that silently adds 4 million BRL of revenue.